# COVID-19 Home Diagnosis AI System
## Using the DEMI Causal Algorithm + Large Language Model (Google Gemini)

**Course:** Comparative Effectiveness  
**Instructor:** Farrokh Alemi, Ph.D.  
**Author:** Gchandanareddy  
**Spring 2026**


In [ ]:
# =========================================================
# STEP 0 — INSTALL DEPENDENCIES
# =========================================================
import subprocess, sys
pkgs = ["networkx", "xgboost", "google-generativeai", "scikit-learn",
        "pandas", "numpy", "matplotlib"]
for pkg in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("All packages installed!")


In [ ]:
# =========================================================
# STEP 1 — IMPORTS
# =========================================================
import pandas as pd
import numpy as np
import itertools
import warnings
import os
import json
import textwrap
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import networkx as nx

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

BOOST_NAME = None
try:
    from xgboost import XGBClassifier
    BOOST_NAME = "XGBoost"
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    BOOST_NAME = "GradientBoosting"

print("All imports successful!")
print(f"Boosting model: {BOOST_NAME}")


In [ ]:
# =========================================================
# STEP 2 — FILE PATHS
# All files must be in the same folder as this notebook.
# =========================================================
RAW_FILE  = "COVIDCARE_FORSUBMISSION_MIT_CLEANED_Phase_II_2021-12-03.csv"
KB_FILE   = "merged.csv"
DICT_FILE = "COVIDCARE_survey_dictionary_v2_ForSubmission_MIT_Phase_II_2021-12-26.csv"

for f in [RAW_FILE, KB_FILE, DICT_FILE]:
    status = "✓ Found" if os.path.exists(f) else "✗ NOT FOUND — upload to this folder"
    print(f"{status}: {f}")


In [ ]:
# =========================================================
# STEP 3 — LOAD FILES
# =========================================================
df         = pd.read_csv(RAW_FILE)
kb         = pd.read_csv(KB_FILE)
dictionary = pd.read_csv(DICT_FILE)

df.columns         = [c.strip() for c in df.columns]
kb.columns         = [c.strip() for c in kb.columns]
dictionary.columns = [c.strip() for c in dictionary.columns]

print("RAW shape:        ", df.shape)
print("KB shape:         ", kb.shape)
print("Dictionary shape: ", dictionary.shape)


In [ ]:
# =========================================================
# STEP 4 — TARGET VARIABLE
# =========================================================
TARGET = "PCR Test Positive"
if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' not found.")

print("Target:", TARGET)
print(df[TARGET].value_counts(dropna=False))


In [ ]:
# =========================================================
# STEP 5 — DEMI STEP 1: TIER ASSIGNMENT (Temporal Order)
# =========================================================
def assign_tier(col_name: str) -> int:
    c = str(col_name).lower()
    if c == "pcr test positive":                                          return 4
    if any(x in c for x in ["pinkline","blueline","pinkblue_confirm",
                              "blue_nopink_confirm","noblue_confirm",
                              "athome","testkit_performing","which_test"]): return 3
    if any(x in c for x in ["vaccine","vacc","flu_shot","covid_vaccine"]): return 1
    if any(x in c for x in ["dob","age","gender","race","ethnicity","birthsex"]): return 0
    return 2

tier_df         = pd.DataFrame({"variable": df.columns})
tier_df["tier"] = tier_df["variable"].apply(assign_tier)
tier_map        = dict(zip(tier_df["variable"], tier_df["tier"]))

print("Tier counts:")
print(tier_df["tier"].value_counts().sort_index())


In [ ]:
# =========================================================
# STEP 6 — RULE 10: Remove self-comparisons
# =========================================================
kb = kb[kb["concept_code"] != kb["target_concept_code"]].copy()
print("KB after Rule 10:", kb.shape)


In [ ]:
# =========================================================
# STEP 7 — ADD TIER INFO TO KNOWLEDGEBASE
# =========================================================
kb["concept_tier"] = kb["concept_code"].map(tier_map).fillna(2).astype(int)
kb["target_tier"]  = kb["target_concept_code"].map(tier_map).fillna(2).astype(int)
print("Tier columns added.")
print(kb[["concept_code","concept_tier","target_concept_code","target_tier"]].head())


In [ ]:
# =========================================================
# STEP 8 — DEMI STEP 2: TOTAL EFFECT (Temporal Rules 6-9)
# =========================================================
def apply_temporal_rules(row):
    n11 = row["n_code_target"]
    n10 = row["n_code_no_target"]
    n01 = row["n_target_no_code"]
    ct  = row["concept_tier"]
    tt  = row["target_tier"]

    if n11 == 0:   # Rule 9
        row["n_code_before_target_final"] = n10
        row["n_target_before_code_final"] = n01
        return row
    if ct < tt:    # Rule 6a
        row["n_code_before_target_final"] = n11
        row["n_target_before_code_final"] = 0
        return row
    if ct > tt:    # Rule 6b
        row["n_code_before_target_final"] = 0
        row["n_target_before_code_final"] = n11
        return row
    # Rules 7/8: Same tier
    if pd.notnull(row.get("n_code_before_target", np.nan)) and        pd.notnull(row.get("n_target_before_code", np.nan)):
        row["n_code_before_target_final"] = row["n_code_before_target"]
        row["n_target_before_code_final"] = row["n_target_before_code"]
    else:
        row["n_code_before_target_final"] = n11
        row["n_target_before_code_final"] = n11
    return row

kb = kb.apply(apply_temporal_rules, axis=1)
print("Temporal rules applied.")


In [ ]:
# =========================================================
# STEP 9 — PAIRWISE ASSOCIATION MEASURES
# =========================================================
kb["n11"] = kb["n_code_target"]
kb["n10"] = kb["n_code_no_target"]
kb["n01"] = kb["n_target_no_code"]
kb["n00"] = kb["n_no_code_no_target"]
kb["N"]   = kb[["n11","n10","n01","n00"]].sum(axis=1)

kb["odds_ratio"]     = ((kb["n11"]+0.5)*(kb["n00"]+0.5)) / ((kb["n10"]+0.5)*(kb["n01"]+0.5))
kb["log_odds_ratio"] = np.log(kb["odds_ratio"])

num = kb["n11"]*kb["n00"] - kb["n10"]*kb["n01"]
den = np.sqrt((kb["n11"]+kb["n10"])*(kb["n01"]+kb["n00"])*
              (kb["n11"]+kb["n01"])*(kb["n10"]+kb["n00"]))
kb["phi"] = np.where(den == 0, np.nan, num / den)

kb["support_both"]           = kb["n11"] / kb["N"]
kb["p_target_given_code"]    = np.where((kb["n11"]+kb["n10"])==0, np.nan,
                                         kb["n11"]/(kb["n11"]+kb["n10"]))
kb["p_target_given_no_code"] = np.where((kb["n01"]+kb["n00"])==0, np.nan,
                                         kb["n01"]/(kb["n01"]+kb["n00"]))

pairwise_assoc = kb[[
    "concept_code","target_concept_code","concept_tier","target_tier",
    "n11","n10","n01","n00","support_both","odds_ratio","log_odds_ratio","phi",
    "p_target_given_code","p_target_given_no_code",
    "n_code_before_target_final","n_target_before_code_final"
]].copy()
pairwise_assoc.to_csv("pairwise_associations.csv", index=False)
print("Saved: pairwise_associations.csv")


In [ ]:
# =========================================================
# STEP 10 — CO-OCCURRENCE FREQUENCIES
# =========================================================
pair_freq = kb[["concept_code","target_concept_code","n_code_target"]].copy()
pair_freq = pair_freq.sort_values("n_code_target", ascending=False)
pair_freq.to_csv("pairwise_frequencies.csv", index=False)
print("Top co-occurring pairs:")
print(pair_freq.head(10).to_string(index=False))


In [ ]:
# =========================================================
# STEP 11 — PREPARE MODELING DATA
# =========================================================
def is_admin(col):
    c = str(col).lower()
    if c == TARGET.lower(): return False
    return any(x in c for x in ["date","submission","confirmation",
                                  "internal id","cohort","_deid",
                                  "name","phone","email"])

model_df = df[[c for c in df.columns if not is_admin(c)]].copy()
if TARGET not in model_df.columns:
    model_df[TARGET] = df[TARGET]

for col in model_df.columns:
    if model_df[col].dtype == "object":
        model_df[col] = model_df[col].astype("category").cat.codes.replace(-1, np.nan)
    if str(model_df[col].dtype) == "bool":
        model_df[col] = model_df[col].astype(int)

model_df[TARGET] = pd.to_numeric(model_df[TARGET], errors="coerce")
model_df = model_df.dropna(subset=[TARGET]).copy()
model_df[TARGET] = model_df[TARGET].astype(int)

print("Model data shape:", model_df.shape)
print("PCR Positive:", model_df[TARGET].sum(), "of", len(model_df))


In [ ]:
# =========================================================
# STEP 12 — FEATURE MATRIX WITH INTERACTIONS
# =========================================================
predictor_cols    = [c for c in model_df.columns if c != TARGET and assign_tier(c) < 4]
X_base            = model_df[predictor_cols].copy()
y                 = model_df[TARGET].copy()

# Force all columns to numeric to prevent TypeError
for col in X_base.columns:
    X_base[col] = pd.to_numeric(X_base[col], errors="coerce").fillna(0).astype(float)

symptom_home_vars = [c for c in predictor_cols if assign_tier(c) in [2, 3]]
pair_base         = symptom_home_vars[:20]
triple_base       = symptom_home_vars[:8]

X = X_base.copy()
for a, b in itertools.combinations(pair_base, 2):
    X[f"{a}__X__{b}"] = X[a].astype(float) * X[b].astype(float)
for a, b, c in itertools.combinations(triple_base, 3):
    X[f"{a}__X__{b}__X__{c}"] = X[a].astype(float) * X[b].astype(float) * X[c].astype(float)

print("Feature matrix:", X.shape)


In [ ]:
# =========================================================
# STEP 13 — TRAIN / TEST SPLIT
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print("Train:", X_train.shape, "| Test:", X_test.shape)


In [ ]:
# =========================================================
# STEP 14 — DEFINE & TRAIN MODELS
# =========================================================
prep_scaled = ColumnTransformer([("num", Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("scaler",  StandardScaler(with_mean=False))
]), X.columns.tolist())])

prep_unscaled = ColumnTransformer([("num", Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent"))
]), X.columns.tolist())])

log_model   = Pipeline([("prep", prep_scaled),
    ("model", LogisticRegression(penalty="l2", solver="liblinear",
                                  max_iter=5000, class_weight="balanced"))])
lasso_model = Pipeline([("prep", prep_scaled),
    ("model", LogisticRegressionCV(penalty="l1", solver="saga", cv=5,
                                    max_iter=5000, scoring="roc_auc",
                                    class_weight="balanced", n_jobs=-1))])
if BOOST_NAME == "XGBoost":
    boost_model = Pipeline([("prep", prep_unscaled),
        ("model", XGBClassifier(n_estimators=200, max_depth=4,
                                 learning_rate=0.05, subsample=0.8,
                                 colsample_bytree=0.8, eval_metric="logloss",
                                 random_state=42))])
else:
    boost_model = Pipeline([("prep", prep_unscaled),
        ("model", GradientBoostingClassifier(n_estimators=200,
                                              learning_rate=0.05,
                                              max_depth=3, random_state=42))])

models = {"Logistic": log_model, "LASSO": lasso_model, BOOST_NAME: boost_model}

def mcfadden_r2(y_true, prob):
    prob    = np.clip(prob, 1e-8, 1-1e-8)
    ll_mod  = -log_loss(y_true, prob, normalize=False)
    p_null  = np.clip(np.repeat(np.mean(y_true), len(y_true)), 1e-8, 1-1e-8)
    ll_null = -log_loss(y_true, p_null, normalize=False)
    return 1 - (ll_mod / ll_null)

results, fitted_models = [], {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    fitted_models[name] = model
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    r2   = mcfadden_r2(y_test, prob)
    results.append({"Model": name,
                    "AUC":      round(roc_auc_score(y_test, prob), 4),
                    "Accuracy": round(accuracy_score(y_test, pred), 4),
                    "McFadden_R2": round(r2, 4),
                    "Percent_Variation_Explained": round(r2*100, 2)})

results_df = pd.DataFrame(results).sort_values("AUC", ascending=False)
print("\n--- MODEL RESULTS ---")
print(results_df.to_string(index=False))
results_df.to_csv("model_results.csv", index=False)


In [ ]:
# =========================================================
# STEP 15 — DEMI STEP 3: DIRECT EFFECT (LASSO coefficients)
# =========================================================
lasso_coef = fitted_models["LASSO"].named_steps["model"].coef_[0]
coef_df    = pd.DataFrame({"variable": X.columns, "coefficient": lasso_coef})
direct_predictors = coef_df[coef_df["coefficient"] != 0].copy()
direct_predictors["abs_coef"] = direct_predictors["coefficient"].abs()
direct_predictors = direct_predictors.sort_values("abs_coef", ascending=False)

print("--- TOP 15 DIRECT PREDICTORS ---")
print(direct_predictors.head(15).to_string(index=False))
direct_predictors.to_csv("direct_predictors_pcr.csv", index=False)


In [ ]:
# =========================================================
# STEP 16 — MARKOV BLANKET (parent regressions)
# =========================================================
def fit_parent_model(response_var):
    response_tier = assign_tier(response_var)
    parents = [c for c in model_df.columns
               if c != response_var and assign_tier(c) < response_tier]
    if len(parents) == 0: return None, None

    temp_df = model_df[parents + [response_var]].copy()
    temp_df[response_var] = pd.to_numeric(temp_df[response_var], errors="coerce")
    temp_df = temp_df.dropna(subset=[response_var]).copy()
    if temp_df.shape[0] < 10: return None, None
    temp_df[response_var] = temp_df[response_var].astype(int)
    if not set(temp_df[response_var].dropna().unique()).issubset({0,1}): return None, None

    Xp, yp = temp_df[parents].copy(), temp_df[response_var].copy()
    Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(
        Xp, yp, test_size=0.25, random_state=42, stratify=yp)
    if yp_tr.nunique() < 2 or yp_te.nunique() < 2: return None, None

    imp   = SimpleImputer(strategy="most_frequent")
    Xp_tr = pd.DataFrame(imp.fit_transform(Xp_tr), columns=Xp.columns, index=Xp_tr.index)
    Xp_te = pd.DataFrame(imp.transform(Xp_te),     columns=Xp.columns, index=Xp_te.index)

    m = LogisticRegressionCV(penalty="l1", solver="saga", cv=5, max_iter=5000, n_jobs=-1)
    m.fit(Xp_tr, yp_tr)
    r2 = mcfadden_r2(yp_te, m.predict_proba(Xp_te)[:,1])
    cd = pd.DataFrame({"parent": parents, "coef": m.coef_[0]})
    cd = cd[cd["coef"] != 0].copy()
    cd["abs_coef"] = cd["coef"].abs()
    return cd.sort_values("abs_coef", ascending=False), r2

markov_results = {}
for var in direct_predictors["variable"].head(15):
    if var in model_df.columns:
        coef, r2 = fit_parent_model(var)
        if coef is not None and not coef.empty:
            markov_results[var] = {"coef": coef, "r2": r2}
            print(f"Response: {var}  |  McFadden R2: {r2:.4f}")


In [ ]:
# =========================================================
# STEP 17 — CAUSAL NETWORK DIAGRAM
# =========================================================
G = nx.DiGraph()
for rv, result in markov_results.items():
    for _, row in result["coef"].iterrows():
        G.add_edge(row["parent"], rv)
for var in direct_predictors["variable"].head(15):
    G.add_edge(var, TARGET)

def clean_name(name):
    name = str(name)
    if name == TARGET:                       return "PCR_Positive"
    if "covid_tst_symptoms" in name:         return "symptom_" + name.split("-")[-1]
    if "pinkblue" in name:                   return "home_test_pos"
    if "blue_nopink" in name:               return "home_test_neg"
    if "vaccine" in name.lower():            return "vaccine"
    if "age" in name.lower():                return "age"
    if "gender" in name.lower():             return "gender"
    short = name.replace("30141-","").replace("covid_","").replace("tst_","")
    return "\n".join(textwrap.wrap(short[:20], width=12))

labels = {node: clean_name(node) for node in G.nodes()}

def tier_layout(graph, x_gap=6, y_gap=1.8):
    pos, tn = {}, {}
    for node in graph.nodes():
        tn.setdefault(assign_tier(node), []).append(node)
    for tier in sorted(tn):
        nodes  = sorted(tn[tier])
        center = (len(nodes)-1)/2
        for i, node in enumerate(nodes):
            pos[node] = (tier*x_gap, (center-i)*y_gap)
    return pos

pos         = tier_layout(G)
color_map   = {0:"lightgreen",1:"khaki",2:"skyblue",3:"plum",4:"tomato"}
node_colors = [color_map.get(assign_tier(n),"lightgray") for n in G.nodes()]

plt.figure(figsize=(20,12))
nx.draw_networkx_edges(G, pos, arrows=True, alpha=0.55, width=1.5,
                        arrowstyle="-|>", arrowsize=14,
                        connectionstyle="arc3,rad=0.08")
nx.draw_networkx_nodes(G, pos, node_size=2200, node_color=node_colors,
                        edgecolors="black", linewidths=0.8)
nx.draw_networkx_labels(G, pos, labels=labels, font_size=9, font_weight="bold")

tier_titles = {0:"Tier 0\nDemographics",1:"Tier 1\nVaccination",
               2:"Tier 2\nSymptoms",3:"Tier 3\nAt-home Test",4:"Tier 4\nPCR"}
max_y = max(y for _,y in pos.values()) if pos else 0
for tier, title in tier_titles.items():
    plt.text(tier*6, max_y+2, title, ha="center", va="bottom",
             fontsize=11, fontweight="bold")

plt.title("COVID-19 Causal Diagnostic Network — DEMI Algorithm", fontsize=16)
plt.axis("off")
plt.tight_layout()
plt.savefig("covid_network.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: covid_network.png")


---
## 🤖 LLM Integration — Google Gemini
### Interpreting COVID-19 Results in Plain English

**Setup:** Paste your free API key from aistudio.google.com into the cell below.


In [ ]:
# =========================================================
# STEP 18 — GOOGLE GEMINI SETUP
# Get your free key at: aistudio.google.com
# =========================================================
import google.generativeai as genai

GEMINI_API_KEY = "YOUR_GEMINI_API_KEY_HERE"  # paste your key here

genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("gemini-1.5-flash")

print("Gemini model ready!")


In [ ]:
# =========================================================
# STEP 19 — PREDICT + LLM INTERPRETATION FUNCTION
# =========================================================
best_model_name = results_df.iloc[0]["Model"]
best_model      = fitted_models[best_model_name]
print(f"Best model: {best_model_name} (AUC = {results_df.iloc[0]['AUC']})")

def predict_and_interpret(symptoms_dict: dict,
                           vaccinated: bool = False,
                           at_home_positive: bool = False,
                           patient_description: str = "") -> dict:
    """
    Runs DEMI model on symptom inputs, then uses Gemini LLM
    to interpret the COVID-19 probability in plain English.
    """
    # Build feature row
    row = pd.DataFrame(0, index=[0], columns=X.columns)
    for code, val in symptoms_dict.items():
        if code in row.columns:
            row.loc[0, code] = float(val)
    if vaccinated:
        for col in row.columns:
            if "vaccine" in col.lower():
                row.loc[0, col] = 1.0
    if at_home_positive:
        for col in row.columns:
            if "pinkblue_confirm" in col.lower():
                row.loc[0, col] = 1.0

    # Ensure float dtype
    row = row.astype(float)

    # Get probability from DEMI model
    prob = best_model.predict_proba(row)[0, 1]

    # Risk level
    if prob >= 0.75:   risk = "HIGH"
    elif prob >= 0.50: risk = "MODERATE"
    elif prob >= 0.25: risk = "LOW-MODERATE"
    else:              risk = "LOW"

    # Send to Gemini LLM for interpretation
    prompt = f"""You are a medical AI assistant interpreting COVID-19 results.

Patient info:
- Symptoms: {list(symptoms_dict.keys()) if symptoms_dict else "None specified"}
- Vaccinated: {vaccinated}
- At-home test positive: {at_home_positive}
- Description: {patient_description or "None"}

DEMI Model Result:
- COVID-19 Probability: {prob*100:.1f}%
- Risk Level: {risk}

Provide:
1. A friendly 2-3 sentence explanation of what this probability means
2. A specific recommendation for what the patient should do next
3. One sentence about AI diagnosis limitations

Keep it simple and easy to understand for a non-medical person.
"""

    response       = gemini_model.generate_content(prompt)
    interpretation = response.text.strip()

    print(f"\n{'='*60}")
    print(f"  COVID-19 PROBABILITY : {prob*100:.1f}%")
    print(f"  RISK LEVEL           : {risk}")
    print(f"{'='*60}")
    print(f"\n📋 GEMINI LLM INTERPRETATION:")
    print(interpretation)
    print(f"{'='*60}")

    return {"probability": round(float(prob), 4),
            "probability_pct": f"{prob*100:.1f}%",
            "risk_level": risk,
            "model_used": best_model_name,
            "interpretation": interpretation}


In [ ]:
# =========================================================
# STEP 20 — DEMO: Three Patient Examples
# =========================================================

print("EXAMPLE 1: High-risk patient — fever, loss of smell, fatigue, cough")
result1 = predict_and_interpret(
    symptoms_dict = {
        "30141-covid_tst_symptoms-11": 1,  # fever
        "30141-covid_tst_symptoms-16": 1,  # loss of smell
        "30141-covid_tst_symptoms-10": 1,  # fatigue
        "30141-covid_tst_symptoms-6":  1,  # cough
    },
    vaccinated          = True,
    at_home_positive    = False,
    patient_description = "Fever for 2 days, lost sense of smell, exhausted, dry cough"
)

print("\nEXAMPLE 2: Low-risk patient — mild runny nose and headache")
result2 = predict_and_interpret(
    symptoms_dict = {
        "30141-covid_tst_symptoms-21": 1,  # runny nose
        "30141-covid_tst_symptoms-12": 1,  # headache
    },
    vaccinated          = True,
    at_home_positive    = False,
    patient_description = "Mild runny nose and slight headache"
)

print("\nEXAMPLE 3: At-home test positive — fever, chills, body aches")
result3 = predict_and_interpret(
    symptoms_dict = {
        "30141-covid_tst_symptoms-11": 1,  # fever
        "30141-covid_tst_symptoms-5":  1,  # chills
        "30141-covid_tst_symptoms-17": 1,  # muscle aches
        "30141-covid_tst_symptoms-10": 1,  # fatigue
    },
    vaccinated          = False,
    at_home_positive    = True,
    patient_description = "Fever, chills, body aches, fatigue, at-home test positive"
)


In [ ]:
# =========================================================
# FINAL SUMMARY
# =========================================================
print("=== MODEL PERFORMANCE SUMMARY ===")
print(results_df.to_string(index=False))
print("\nOutput files saved:")
for f in ["pairwise_associations.csv","pairwise_frequencies.csv",
          "model_results.csv","direct_predictors_pcr.csv","covid_network.png"]:
    exists = "✓" if os.path.exists(f) else "✗"
    print(f"  {exists} {f}")
